# Forward Pass Debugging: Layer-by-Layer Reference

This notebook performs a forward pass through a .keras model one layer at a time, for the purpose of verifying low-level implementations in C or RISC-V.

At each step, it prints the output of the current layer. These values act as ground truth for validating manual implementations. The output of one layer is passed directly as the input to the next, preserving the inference flow.

This setup helps identify discrepancies between the high-level model and its low-level counterparts, allowing for precise, layer-specific debugging.


In [37]:
import os
import tensorflow as tf
import numpy as np
import random
import struct

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


## 1.0 Load and preprocess the data

In [38]:
(images, labels), _ = tf.keras.datasets.mnist.load_data()
images = images.astype("float32") / 255.0  # Normalize the images to [0, 1]
images = np.expand_dims(images, -1)  # Add channel dimension
labels = tf.keras.utils.to_categorical(labels, 10)  # One-hot encode the labels

## 2.0 Helper Functions

### 2.1 Function to Get a random image and label

Get a random image and label from the dataset. This function is used to generate a random input for the model.

In [39]:

def save_in_riscv_format(x, filename):
    with open(filename, 'w') as f:
        if len(x.shape) == 4:
            _, height, width, channels = x.shape
            for c in range(channels):
                for i in range(height):
                    row = [f"{x[0, i, j, c]:.6f}" for j in range(width)]
                    f.write(f".float " + ", ".join(row) + "\n")
                f.write("\n")
        
        elif len(x.shape) == 2:
            rows, cols = x.shape
            values = [f"{x[i, j]:.6f}" for i in range(rows) for j in range(cols)]
            f.write(f".float " + ", ".join(values) + "\n")
        
        else:
            raise ValueError("Unsupported shape")


def float32_to_float16_hex(value):
    """Convert float32 to IEEE 754 float16 hex string."""
    f16 = np.float16(value)
    u16 = struct.unpack('>H', struct.pack('>e', f16))[0]
    return f"0x{u16:04x}"

def get_random_image(index=None, output_file="random_image.txt"):
    # Randomly select an image and its label
    if index is None:
        index = random.randint(0, len(images) - 1)
    
    image = images[index].squeeze()        # (28, 28)
    image = tf.expand_dims(image, axis=0)  # (1, 28, 28)
    image = tf.expand_dims(image, axis=-1) # (1, 28, 28, 1)
    label = np.argmax(labels[index])       # Get label

    # Hex output file
    base, ext = os.path.splitext(output_file)
    hex_output_file = base + "_hex.txt"

    # Save both formats
    with open(output_file, "w") as f_float, open(hex_output_file, "w") as f_hex:
        for row in image.numpy().squeeze():
            float_row = ".float " + ", ".join(f"{val:.3f}" for val in row)
            hex_row = ".half  " + ", ".join(float32_to_float16_hex(val) for val in row)
            f_float.write(float_row + "\n")
            f_hex.write(hex_row + "\n")
    
    return image, label

## 3.0 Load the model from mnist_cnn_model.keras

In [40]:
model = tf.keras.models.load_model("../models/fp16_mem_optimised_models/mnist_cnn_model_fp16.keras")

## 4.0 Get a random image, label and step through the model layer by layer

In [41]:
# Defining a function, that does all of that. Saves results to file. This has been doen to avoid confusion
def run_inference():
    # Get a random image and its label
    image, label = get_random_image()

    conv2d_out = model.layers[0](image)
    save_in_riscv_format(conv2d_out, "conv2d_out.txt")
    
    relu_out = model.layers[1](conv2d_out)
    save_in_riscv_format(relu_out, "relu_out.txt")

    maxpool_out = model.layers[2](relu_out)
    save_in_riscv_format(maxpool_out, "maxpool_out.txt")

    flatten_out = model.layers[3](maxpool_out)
    save_in_riscv_format(flatten_out, "flatten_out.txt")
    
    dense_out = model.layers[4](flatten_out)
    save_in_riscv_format(dense_out, "dense_out.txt")

    softmax_out = model.layers[5](dense_out)
    save_in_riscv_format(softmax_out, "softmax_out.txt")

    prediction = np.argmax(softmax_out)

    return prediction, label

In [42]:
prediction, label = run_inference()
print(f"Prediction: {prediction}, Label: {label}")


Prediction: 4, Label: 4
